## Loading the temperature forecast: test with Zurich

In stage 2, part B, temperature enters the regression as a feature. To avoid leakage, we do not use the
measured temperature but the **forecast issued 24 hours earlier** (`temperature_2m_previous_day1` from the
Open-Meteo Previous Runs API, model `gfs_seamless`). This information would actually have been available
at forecast time.

Before loading all cities and the full period, we test the request with Zurich for two days in April 2021.
We check which columns are returned, whether the timestamps are in UTC and whether the values look plausible.

In [1]:
import json
import urllib.parse
import urllib.request

import pandas as pd

# Base URL of the Previous Runs API
BASE_URL = "https://previous-runs-api.open-meteo.com/v1/forecast"


def fetch_temperature(lat, lon, start, end):
    # Loads the analysed temperature and the forecast issued 24 h earlier,
    # timestamps in UTC
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": "temperature_2m,temperature_2m_previous_day1",
        "models": "gfs_seamless",
        "start_date": start,
        "end_date": end,
        "timezone": "GMT",
    }
    url = BASE_URL + "?" + urllib.parse.urlencode(params)
    with urllib.request.urlopen(url) as response:
        data = json.load(response)

    # Print the raw structure once to see the format
    print("Top-level keys:", list(data.keys()))
    print("Units:", data["hourly_units"])

    # Convert hourly values to a DataFrame with a UTC index
    df = pd.DataFrame(data["hourly"])
    df["time"] = pd.to_datetime(df["time"], utc=True)
    return df.set_index("time")


# Test: Zurich, two days
zurich = fetch_temperature(47.37, 8.54, "2021-04-01", "2021-04-02")
print(zurich.shape)
zurich.head(8)

Top-level keys: ['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly']
Units: {'time': 'iso8601', 'temperature_2m': '°C', 'temperature_2m_previous_day1': '°C'}
(48, 2)


,temperature_2m,temperature_2m_previous_day1
time,,
2021-04-01 00:00:00+00:00,11.3,10.9
2021-04-01 01:00:00+00:00,10.9,10.5
2021-04-01 02:00:00+00:00,10.3,10.1
2021-04-01 03:00:00+00:00,9.9,10.2
2021-04-01 04:00:00+00:00,9.3,9.9
2021-04-01 05:00:00+00:00,9.0,9.8
2021-04-01 06:00:00+00:00,9.5,10.3
2021-04-01 07:00:00+00:00,11.2,11.9


The request works as expected:

- **Format:** one row per hour, two columns: `temperature_2m` (analysed temperature) and
  `temperature_2m_previous_day1` (forecast issued 24 h earlier). Two days give 48 rows.
- **Time:** the index is in UTC and starts at 00:00, matching the UTC grid of the load data.
- **Plausibility:** values between roughly 9 and 12 °C fit a mild early-April night in Zurich.
  The forecast stays close to the analysis but deviates by up to about 0.8 °C, as expected for a real forecast.

One point to settle before merging: the temperature is an instantaneous value at the timestamp, while each
load value is an hourly average labelled at the end of its interval. We will decide on the alignment when
building the full feature table.

Next step: load the full period (April 2021 to end of 2025) for several Swiss cities and build a
population-weighted average.